# **Assignment-1-Ritvik-2025122012**

Things to note:
* Seeding & personalization:
    * Need to generate seed from the SHA-256 hash of IIITH username (part before @), mod 2³².
    * Use this seed in every RNG call throughout (np.random.default_rng(seed) etc.).
    * Every plot must show username in the title/filename (using plt.text, refer template).
* Code/plotting requirements:
    * Object-oriented programming required.
    * Every plot needs title, x-label, y-label, and legend (if applicable); missing labels/legend = 0 marks on that answer.
    * Visualization logic and computation logic needs to be kept in separate functions.
    * Docstrings required on all methods.
* Library restrictions:
    * Section 1 (AES/KNN): no NLP libraries, pretrained models, language models, or other libraries/models specifically designed to extract features (only for feature extraction, 1.1); KNN must be implemented from scratch.
    * Section 2 ( unless a sub-question says otherwise):
        * np.random and random banned everywhere so must use your own LCG (Q2.1.1) instead.
        * scipy.stats banned inside every implemented function.
        * scipy.special utilities (erfinv, gammaln, comb, betaincinv) allowed only where explicitly permitted per sub-question.
        * sklearn (or anything implementing Naive Bayes/KNN/metrics) banned throughout, except Q2.3.4(d), where it's used as comparison alone.
* Data handling specifics
    * 80:10:10 train/val/test split and must report sample counts per split.
    * Z-score normalization params (µ, σ) must be learned from training data only and then applied to val/test.
    * At least 4 meaningful feature visualizations are required.
    * Report validation performance across all K & distance-metric combinations in a table, then identify/justify the best config before final test evaluation.
    * Spam classifier: fit word/class frequencies on train split only. The test data must be untouched until evaluation.
    * Various functions have very specific signatures/return types mentioned (e.g., MLE estimators must return plain Python floats, functions must dispatch generically over breed/feature counts rather than hardcoding) which must be followed exactly.
* Also check if following si included as they are req for sub-questions:
    * 1.1 -> the feature motivation/interpretation explained
    * 1.3.2 -> K and distance-metric trend 
    * 2.1.5 / 2.1.4(d) -> CDF validation done and reported
    * 2.2.4 -> confusion matrix + P/R/F1 computed on test set
    * 2.4.1 -> posterior vs MLE comparison + prior behavior report

In [22]:
import hashlib
username="sairitvik.uppuganti"
seed = int(hashlib.sha256(username.encode()).hexdigest(), 16) % (2**32)
print(seed)

1703431610


## **Imports:**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import string
import math

# **1. Automated Essay Scoring System**

## 1.1 Feature Engineering

In [26]:
df = pd.read_csv('essays.csv')
display(df.head())

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State SenatorThis is a letter to argue i...",3


In [ ]:
class EssayFeatureExtractor:
    def __init__(self):
        pass

    def extract_features(self, essay):

        words = []
        sentences = []

        raw_words = essay.split()  #we first get words in essay by splitting whitespace
        for word in raw_words:
            word = word.strip(string.punctuation)  #strip the punctuations
            word = word.lower()     #make it lowercase so thst "Word" and "word" are same
            if word: 
                    words.append(word)  

        raw_sentences = essay.replace('?', '.').replace('!', '.').split('.')    #same as words but split at '.', '!', '?' instead
        for sentence in raw_sentences:
            sentence = sentence.strip(string.punctuation)   #same as for words
            sentence = sentence.lower()
            if sentence:
                sentences.append(sentence)

        final_features = {} #final feature list comprising of everything extracted

        final_features.update(self.length_fluency(words, sentences))
        final_features.update(self.vocab_richness(words))

        return final_features
        

    def length_fluency(self, words, sentences):       #refer md below
            n_words = max(1, len(words))
            n_sents = max(1, len(sentences))

            features = {}
            features['n_words'] = n_words
            features['n_sents'] = n_sents
            features['avg_words_per_sent'] = n_words / n_sents
            features['avg_word_len'] = sum(len(word) for word in words) / n_words

            return features
                                                
    def vocab_richness(self, words):                  #refer md beloww
            n_words = max(1, len(words))
            word_counts = {}
            for word in words:
                word_counts[word] = word_counts.get(word, 0) + 1

            n_unique_words = len(word_counts)
            root_ttr = n_unique_words / (n_words ** 0.5)

            hapax_count = 0
            for count in word_counts.values():
                if count == 1:
                    hapax_count += 1
            hapax_ratio = hapax_count / n_words

            entropy = 0
            for count in word_counts.values():
                prob = count / n_words
                entropy += -prob * math.log2(prob)

            features = {}
            features['root_ttr'] = root_ttr
            features['hapax_ratio'] = hapax_ratio
            features['entropy'] = entropy

            return features

    def features2dictionary(self, essays):
        feature_dicts = [self.extract_features(essay) for essay in essays]
        return pd.DataFrame(feature_dicts)
        

#### 1. Length Fluency:
* Desc: 
* Extracts the following features: 
    1. Number of words 
    2. Number of sentences
    3. Avg words per sentence
    4. Avg word length

#### 2. Vocabulary Richness:
* Desc: 
* Extracts the following features: 
    1. Root Type-Token Ratio (root_ttr)
    2. Hapax Legomena Ratio
    3. Shannon Entropy over word distribution 

## 1.2 Data Preprocessing


### 1.2.1 (a) Train–Validation–Test Split [3 marks]


In [38]:
extractor = EssayFeatureExtractor()
X = extractor.features2dictionary(df['full_text'])

In [39]:
display(X.head())

,n_words,n_sents,avg_words_per_sent,avg_word_len,root_ttr,hapax_ratio,entropy
0,498,14,35.571429,4.331325,10.575413,0.331325,7.162753
1,328,21,15.619048,3.963415,8.116717,0.250000,6.729865
2,547,25,21.880000,4.510055,9.705819,0.259598,7.122613
3,445,21,21.190476,4.955056,10.523809,0.325843,7.177024
4,368,16,23.000000,4.855978,7.454390,0.225543,6.297999


In [ ]:
y = df['score']

X_train, X_t, y_train, y_t = train_test_split(X, y, test_size=0.2, random_state=seed) # 80:20 for train:test split
X_val, X_test, y_val, y_test = train_test_split(X_t, y_t, test_size=0.5, random_state=seed) # 10:10 for val:test split



In [43]:
display(y.head())

0    3
1    3
2    4
3    4
4    3
Name: score, dtype: int64

### 1.2.2 (b) Feature Normalization

In [57]:
class ZScoreStd:
    def __init__(self): # intialise mean and std
        self.mean = None
        self.stdev = None
    
    def fit(self, X_feat):
        X_feat2 = np.array(X_feat)  #convert to np if in dataframe form
        self.mean = np.mean(X_feat2, axis=0)
        self.stdev = np.std(X_feat2, axis=0)
        return self.mean, self.stdev

    def transform(self, X_feat):        # apply the z score standardisation 
        if self.mean is None or self.stdev is None:
            raise ValueError("Mean and standard deviation have not been computed. Call the 'fit' method first.")
        X_feat2 = np.array(X_feat)
        return (X_feat2 - self.mean) / self.stdev

    def fit_transform(self, X_feat):  #one thing to note is that, we need to apply both fit and transform for training data 
        self.fit(X_feat)                # but only transform for test data, so we have two separate methods for that
        return self.transform(X_feat)

In [59]:
scaler = ZScoreStd()

X_train_f_t = scaler.fit_transform(X_train)
X_val_t = scaler.transform(X_val)
X_test_t = scaler.transform(X_test)

### 1.2.3 (c) Feature Visualization [4 marks]

## 1.3 K-Nearest Neighbors [20 marks]


### 1.3.1 (a) Implementation [15 marks]

### 1.3.2 (b) Hyperparameter Tuning [5 marks]

In [ ]:
class DistanceMetrics:

    def __init__(self): #dont need to save any variables in class so no need for init -> added to make it clear avoid any error later.
        pass

    @staticmethod                                   # added staticmethod decorator since they need to only act as a function (no vars/instance req).
    def euclidean(xi, X_train):                       # xi is one test sample, X_train is training data
        d = np.sqrt(np.sum((X_train - xi) ** 2, axis=1))
        return d

    @staticmethod
    def manhattan(xi, X_train):
        d = np.sum(np.abs(X_train - xi), axis=1)
        return d

    @staticmethod
    def cosine(xi, X_train):
        d = 1 - (np.dot(X_train, xi) / (np.linalg.norm(X_train, axis=1) * np.linalg.norm(xi)))
        return d

    @classmethod                            #classmethod decorator since we need access to methods of class
    def dist(cls, metric_name):           # cls cant access instances but only methods
        metrics = {
            "euclidean": cls.euclidean,
            "manhattan": cls.manhattan,
            "cosine": cls.cosine
        }
        if metric_name.lower() not in metrics:                  # check if invalid name & raise error
            raise ValueError(f"Unknown metric: {metric_name}")
        return metrics[metric_name.lower()]

## 1.4 Evaluation [5 marks]


In [ ]:
class Evaluator:
    def __init__(self, y_true, y_pred):
        self.y_true = np.array(y_true)
        self.y_pred = np.array(y_pred)

    def mean_absolute_error(self):
        mae = np.mean(np.abs(self.y_true - self.y_pred))
        return mae

    def root_mean_squared_error(self):
        rmse = np.sqrt(np.mean((self.y_true - self.y_pred) ** 2))
        return rmse

    def r_squared(self):
        num = np.sum((self.y_true - self.y_pred) ** 2)
        den = np.sum((self.y_true - np.mean(self.y_true)) ** 2)
        r2 = 1 - (num / den)
        if den == 0:
            return 0.0    # edge case if all the true values are the same r^2 isnt defined -> return 0.0 or NaN
        else:
            return r2

    def pearson_correlation(self):
        num = np.sum((self.y_true - np.mean(self.y_true)) * (self.y_pred - np.mean(self.y_pred)))
        den = np.sqrt(np.sum((self.y_true - np.mean(self.y_true)) ** 2) * np.sum((self.y_pred - np.mean(self.y_pred)) ** 2))
        r = num / den
        if den == 0:
            return 0.0  # same edge case as for r^2
        else:
            return r

    def metrics(self):
        return {
            'MAE': self.mean_absolute_error(),
            'RMSE': self.root_mean_squared_error(),
            'R^2': self.r_squared(),
            'Pearson Correlation': self.pearson_correlation()
        }

### 1.4.1 (a) Mean Absolute Error

### 1.4.2 (b) Root Mean Squared Error

### 1.4.3 (c) R2 Score

### 1.4.4 (d) Pearson Correlation Coefficient

# **2. Probability Basics and Bayesian Learning**

## 2.1 A Sampling Toolkit From Scratch

### 2.1.1 Uniform Generator (LCG) [3 marks]

### 2.1.2 Gaussian Sampler [4 marks]

### 2.1.3 Binomial Sampler [5 marks]

### 2.1.4 Image Sampler — Sampling a 2D Empirical Density [6 marks]

### 2.1.5 Generator Validation [6 marks]

## 2.2 Naive Bayes Classifier for Mixed Features [34 marks]

### 2.2.1 Maximum-Likelihood Estimators [7 marks]

### 2.2.2 Model Fitting [8 marks]

### 2.2.3 MAP Prediction [9 marks]

### 2.2.4 Evaluation Metrics [10 marks]

## 2.3 Spam Detection [22 marks]

### 2.3.1 Text Statistics [10 marks]

### 2.3.2 Smoothed Classifier [12 marks]

## 2.4 Bayesian Parameter Estimation [12 marks]

### 2.4.1 Bayesian Estimation of a Bark-Day Rate [12 marks]

#### (a)

#### (b)

#### (c)